# Anomaly detection — exploration notebook

What this notebook demonstrates, in a few minutes and without retraining anything:

1. **The prepared data** — the seeded synthetic telemetry stream every detector
   in this project is scored on, loaded from persisted artifacts (not regenerated).
2. **The headline comparison** — the full run ladder (untuned + tuned detectors +
   seeded random baseline) as persisted in `reports/metrics.json`.
3. **One key result, reproduced** — re-scoring the persisted untuned
   IsolationForest and the seeded random baseline on the held-out test window,
   matching `reports/metrics.json` bit-for-bit.

The point of the project is the evaluation harness — time-ordered split, fixed
a-priori alert threshold, seeded random baseline — not any single detector.

## How to run

```bash
uv sync --all-packages   # from the repository root
```

Then run the cells from the **project directory** (`projects/03-anomaly-detection`).

Everything here reads persisted artifacts from `data/` and `reports/`, produced by
the pipeline (`uv run dvc repro` from the project dir regenerates them end-to-end).
No cell downloads data, hits the network, or retrains a detector — each step takes
seconds. Outputs are committed cleared by convention (exploratory only).

## 1. The prepared data

The default dataset is seeded synthetic telemetry: 20,000 one-minute rows on three
channels (`cpu`, `mem`, `latency_ms`) with 2% of minutes injected as anomalies. The
ground-truth `is_anomaly` flag is used for **evaluation only** — no detector sees it.
A versioned `DatasetContract` (checksum, source, license) travels with the parquet.

In [ ]:
from anomaly_detection.config import AnomalyConfig
from anomaly_detection.data import load_raw

config = AnomalyConfig()
raw = load_raw(config)

print(f"rows: {raw.height:,}  columns: {raw.width}")
print(f"injected anomaly rate: {raw['is_anomaly'].mean():.2%}")
print(f"dataset contract: {config.contract_path.name} (data/{config.contract_path.name})")
raw.head(5)

## 2. The headline comparison — run ladder

`reports/metrics.json` holds ROC-AUC / PR-AUC for every run on the held-out test
window (the last 30% of the timeline). PR-AUC is the metric that matters at ~1.9%
prevalence, where a naive "everything is normal" classifier already scores 0.019.

Read the persisted ladder and rank by PR-AUC: the tuned One-Class SVM is the
precision-recall champion, with untuned IsolationForest close behind and LOF at
chance — its failure is structural, not hyperparametric.

In [ ]:
import json

import polars as pl

metrics = json.loads((config.reports_path / "metrics.json").read_text())
ladder = pl.DataFrame(
    {
        "run": list(metrics),
        "roc_auc": [m["roc_auc"] for m in metrics.values()],
        "pr_auc": [m["pr_auc"] for m in metrics.values()],
    }
).sort("pr_auc", descending=True)
ladder

## 3. One key result, reproduced — model vs. random baseline

The admission fee for any detector here is beating the **seeded random baseline**.
Reproduce the two anchor rows of the ladder — untuned IsolationForest and random —
by loading the persisted feature matrix and fitted detector, scoring the test
window, and comparing against `metrics.json`. Everything comes from `data/`
artifacts; nothing is refit.

In [ ]:
from anomaly_detection.evaluate import evaluate_scores, random_scores
from anomaly_detection.features import load_features
from anomaly_detection.model import anomaly_scores, feature_matrix, load_model, split_train_test

df = load_features(config)
_, test = split_train_test(df, config.train_frac)  # time-ordered: last 30% is test
x_test = feature_matrix(test, config)
y_test = test["is_anomaly"].to_numpy().astype(int)

scores = {
    "isolation_forest": anomaly_scores(
        load_model("isolation_forest", config), "isolation_forest", x_test
    ),
    "random": random_scores(len(y_test), config.seed),
}

for name, s in scores.items():
    run = evaluate_scores(name, y_test, s)
    roc = run.get("roc_auc").value
    pr = run.get("pr_auc").value
    print(f"{name:>20}  ROC-AUC {roc:.4f}  PR-AUC {pr:.4f}")
    assert abs(roc - metrics[name]["roc_auc"]) < 1e-9
    assert abs(pr - metrics[name]["pr_auc"]) < 1e-9

print("matches reports/metrics.json — the persisted artifacts reproduce end-to-end")

## Where to go next

- **Case study** — the full narrative (EDA, tuning, guardrails, business framing):
  [/projects/03-anomaly-detection](/projects/03-anomaly-detection/)
- **DESIGN.md** — architecture, ADRs, and the evaluation contract:
  [../DESIGN.md](../DESIGN.md)
- **Code** — `src/anomaly_detection/` (`data.py`, `features.py`, `model.py`,
  `hpo.py`, `evaluate.py`, `stream.py`, `probes.py`); entry point is the `dvc.yaml`
  pipeline, runnable end-to-end with `uv run dvc repro`.